# Alignment of volume stacks

## Alignment using Kornia
Kornia uses GPU acceleration and intensity-based global optimization between image pairs

In [ ]:
import os
from pathlib import Path
import torch
import kornia
import tifffile
import numpy as np
from dotenv import load_dotenv

load_dotenv("../.env")
DATA_ROOT = Path(os.environ.get("DATA_ROOT"))

In [ ]:
def load_image(path: Path) -> torch.Tensor:
    """Load image to floating point tensor"""
    img = tifffile.imread(path)
    tensor = kornia.image_to_tensor(img, keepdim=False).float()
    return tensor


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device {device}")

path_img_fixed = DATA_ROOT / Path("raw/Au_01-vol_01/Au_01-vol_01-z_0837.tif")
path_img_moving = DATA_ROOT / Path("raw/Au_01-vol_01/Au_01-vol_01-z_0838.tif")

img_fixed = load_image(path_img_fixed).to(device)
img_moving = load_image(path_img_moving).to(device)

registrator = kornia.geometry.ImageRegistrator(
    model_type=kornia.geometry.Similarity(scale=False).to(device),
    loss_fn=torch.nn.functional.mse_loss,
    warper=kornia.geometry.HomographyWarper,
).to(device)

transformation = registrator.register(
    img_fixed,
    img_moving,
    verbose=True,
)

Using devide cpu
Loss = 159.2874, iter=0
Loss = 49.8463, iter=10
Loss = 37.0194, iter=20
Loss = 36.6424, iter=30
Loss = 36.3613, iter=40
Loss = 36.2056, iter=50
Loss = 36.0395, iter=60
Loss = 35.9661, iter=70
Loss = 35.9315, iter=80
Loss = 96.4705, iter=0
Loss = 96.3005, iter=10
Loss = 96.2633, iter=20
Loss = 96.2655, iter=30
Loss = 242.6365, iter=0
Loss = 242.3850, iter=10
Loss = 242.3954, iter=20
Loss = 242.4913, iter=30
Loss = 242.4053, iter=40
Loss = 242.3815, iter=50
Loss = 242.3804, iter=60
Loss = 599.0938, iter=0
Loss = 604.6931, iter=10
Loss = 611.6909, iter=20
Loss = 615.3974, iter=30
Loss = 600.2649, iter=40
Loss = 599.6435, iter=50
Loss = 598.9544, iter=60
Loss = 598.7175, iter=70
Loss = 599.1447, iter=80
Loss = 598.4579, iter=90
Loss = 3465.7017, iter=0
Loss = 3467.5122, iter=10
Loss = 3548.8018, iter=20
Loss = 3706.4897, iter=30
Loss = 3592.8904, iter=40
Loss = 3605.2632, iter=50
Loss = 3481.5337, iter=60
Loss = 3478.8315, iter=70
Loss = 3512.3740, iter=80
Loss = 3787.7979

In [13]:
with torch.no_grad():
    img_out = kornia.geometry.homography_warp(
        img_moving, transformation, img_fixed.shape[-2::]
    )
    img_out = img_out.detach().cpu().numpy().astype(np.int8).squeeze()

In [14]:
tifffile.imwrite(path_img_moving.name, img_out)